# Maximum usable PEM discharge power

Goal: estimate the maximum PEM power that is usable by the EMS.

I use the polarization/load sweep because it directly shows voltage, current, and power at different
loads. The maximum usable power is not the largest absolute power in the sweep. It is the largest
power while the PEM voltage is still above the voltage cutoff.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This makes the notebook work both from the repo root and from its own folder.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find the project root folder containing data/")
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)


## 1. Load and inspect the polarization sweep


In [ ]:
sweep_file = PROJECT_ROOT / "data/PEM_test/current_sweep/PEM_polarization_characteristics.csv"
sweep_raw = pd.read_csv(sweep_file)

print("Rows:", len(sweep_raw))
display(sweep_raw.head())


## 2. Correct voltage and current and select discharge data


In [ ]:
sweep = sweep_raw.copy()
sweep["timestamp"] = pd.to_datetime(sweep["timestamp"])
sweep["time_s"] = (sweep["timestamp"] - sweep["timestamp"].iloc[0]).dt.total_seconds()
sweep["pem_voltage_V"] = sweep["ina4_bus_V"] - 0.064
sweep["pem_current_A"] = 0.843 * (sweep["ina4_current_mA"] / 1000) + 0.001
sweep["load_current_A"] = (-sweep["pem_current_A"]).clip(lower=0)
sweep["pem_power_W"] = sweep["pem_voltage_V"] * sweep["load_current_A"]

discharge = sweep[(sweep["scenario"] == 6) & (sweep["load_current_A"] > 0)].copy()
discharge["discharge_time_s"] = discharge["time_s"] - discharge["time_s"].iloc[0]

display(discharge[["discharge_time_s", "pem_voltage_V", "load_current_A", "pem_power_W"]].head())


## 3. Average the sweep into load steps


In [ ]:
STEP_LENGTH_S = 10
discharge["step_index"] = (discharge["discharge_time_s"] // STEP_LENGTH_S).astype(int)

step_summary = (
    discharge.groupby("step_index")
    .agg(
        current_A=("load_current_A", "mean"),
        voltage_V=("pem_voltage_V", "mean"),
        power_W=("pem_power_W", "mean"),
    )
    .reset_index()
)
step_summary["power_mW"] = step_summary["power_W"] * 1000

display(step_summary.head(15))


## 4. Apply the voltage cutoff


In [ ]:
PEM_MIN_USABLE_VOLTAGE = 0.54975

# The sweep later turns back down after collapse. I only keep the increasing-current branch.
step_summary["is_increasing_current"] = step_summary["current_A"].diff().fillna(0) >= 0
step_summary["is_above_cutoff"] = step_summary["voltage_V"] >= PEM_MIN_USABLE_VOLTAGE
step_summary["is_usable"] = step_summary["is_increasing_current"] & step_summary["is_above_cutoff"] & (step_summary["power_W"] > 0)

usable_steps = step_summary[step_summary["is_usable"]].copy()
best_usable = usable_steps.loc[usable_steps["power_W"].idxmax()]
absolute_best = step_summary.loc[step_summary["power_W"].idxmax()]

print("Best usable step:")
display(best_usable.to_frame().T)

print("Absolute best power step, shown for comparison:")
display(absolute_best.to_frame().T)


## 5. Plot why the EMS limit is lower than the absolute maximum


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(step_summary["current_A"], step_summary["power_W"], marker="o", label="all sweep steps")
plt.scatter(best_usable["current_A"], best_usable["power_W"], color="red", zorder=5, label="selected EMS limit")
plt.xlabel("Load current [A]")
plt.ylabel("PEM power [W]")
plt.title("Selected maximum usable PEM power")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(step_summary["current_A"], step_summary["voltage_V"], marker="o", label="voltage")
plt.axhline(PEM_MIN_USABLE_VOLTAGE, color="black", linestyle="--", label="voltage cutoff")
plt.scatter(best_usable["current_A"], best_usable["voltage_V"], color="red", zorder=5, label="selected EMS limit")
plt.xlabel("Load current [A]")
plt.ylabel("PEM voltage [V]")
plt.title("PEM voltage decides which power points are usable")
plt.grid(True)
plt.legend()
plt.show()


## 6. Values to use in the app


In [ ]:
PEM_MAX_DISCHARGE_CURRENT_A = best_usable["current_A"]
PEM_MAX_DISCHARGE_POWER_W = best_usable["power_W"]

pem_power_parameters = pd.DataFrame(
    {
        "parameter": [
            "PEM_MIN_USABLE_VOLTAGE",
            "PEM_MAX_DISCHARGE_CURRENT_A",
            "PEM_MAX_DISCHARGE_POWER_W",
        ],
        "value": [
            PEM_MIN_USABLE_VOLTAGE,
            PEM_MAX_DISCHARGE_CURRENT_A,
            PEM_MAX_DISCHARGE_POWER_W,
        ],
        "unit": ["V", "A", "W"],
        "meaning": [
            "Fuel cell voltage cutoff",
            "Current at highest usable sweep point",
            "Highest power above cutoff on the increasing-current branch",
        ],
    }
)

display(pem_power_parameters)
